In [27]:
# Noah Lebowitz-Lockard
# Tuesday, May 5, 2026

# final_birdclef_model.ipynb

# This model combines all of the previous models we made. On top of that, it classifies the rare classes separately.

import librosa
import numpy as np
import pandas as pd
import pickle
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
TEST_DIR = "/kaggle/input/birdclef-2026/test_soundscapes"
#MODEL_DIR = "/kaggle/input/birdclef-2026"

# animal_model determines the likelihood that the file has animals from each class (animal_class_classifier.ipynb).

animal_model = tf.keras.models.load_model(
    f"best_multilabel_model.keras",
    compile=False
)

# The next four models determine the likelihood that the file has animals from the most common species in each animal class.
# ("aves_classifier_gpu.ipynb", "amphibian_classifier.ipynb", "insecta_classifier.ipynb", "mammalia_classifier.ipynb")
# For insects, Species 0 (244024) was far more common that the other 14 species, so we trained a separate model for it.

common_bird_model = tf.keras.models.load_model(f"best_common_bird_model_gpu.keras", compile=False)
common_amphibian_model = tf.keras.models.load_model(f"best_common_amphibian_model.keras", compile=False)
common_insect0_model = tf.keras.models.load_model(f"best_insect0_model.keras", compile=False)
common_insect1_model = tf.keras.models.load_model(f"best_insect1_model.keras", compile=False)
common_mammal_model = tf.keras.models.load_model(f"best_common_mammal_model.keras", compile=False)

In [31]:
import tensorflow as tf

files = {
    "best_multilabel_model.keras": "best_multilabel_model.h5",
    "best_common_bird_model_gpu.keras": "common_bird_model.h5",
    "best_common_amphibian_model.keras": "common_amphibian_model.h5",
    "best_insect0_model.keras": "common_insect0_model.h5",
    "best_insect1_model.keras": "common_insect1_model.h5",
    "best_common_mammal_model.keras": "common_mammal_model.h5",
}

for old, new in files.items():
    model = tf.keras.models.load_model(old, compile=False)
    model.save(new)
    print("Saved", new)

Saved best_multilabel_model.h5
Saved common_bird_model.h5
Saved common_amphibian_model.h5
Saved common_insect0_model.h5
Saved common_insect1_model.h5


Saved common_mammal_model.h5


In [50]:
# Saves the weights of each model to a .h5 file, which can be loaded in the future with model.load_weights("filename.h5").
# This is much smaller than saving the entire model, and we can still load the weights into the same architecture later.

files = {
    "best_multilabel_model.keras": "best_multilabel_model.weights.h5",
    "best_common_bird_model_gpu.keras": "common_bird_model.weights.h5",
    "best_common_amphibian_model.keras": "common_amphibian_model.weights.h5",
    "best_insect0_model.keras": "common_insect0_model.weights.h5",
    "best_insect1_model.keras": "common_insect1_model.weights.h5",
    "best_common_mammal_model.keras": "common_mammal_model.weights.h5",
}

for old, new in files.items():
    model = tf.keras.models.load_model(old, compile=False)
    model.save_weights(new)

In [15]:
mammal_feature_model = tf.keras.Model(
    inputs=common_mammal_model.inputs,
    outputs=common_mammal_model.layers[-2].output
)

bird_feature_model = tf.keras.Model(
    inputs=common_bird_model.inputs,
    outputs=common_bird_model.layers[-2].output
)

insect_feature_model = tf.keras.Model(
    inputs=common_insect1_model.inputs,
    outputs=common_insect1_model.layers[-2].output
)

amphibian_feature_model = tf.keras.Model(
    inputs=common_amphibian_model.inputs,
    outputs=common_amphibian_model.layers[-2].output
)

In [16]:
# Here are all of the animal species. The only reptile is 116570.

common_amphibians = ['1491113', '22961', '22967', '22973', '23158', '24279', '24321', '25092', '326272', '517063', '555146', '65377', '65380']
common_amphibians += ['66971']
rare_amphibians = ['1176823', '1595929', '22930', '22956', '22983', '22985', '23150', '23154', '23176', '23724', '24285', '24287', '25073']
rare_amphibians += ['25214', '476521', '555123', '555145', '64898', '67107', '67252', '70711']
common_birds = ['astcra1', 'baffal1', 'banana', 'barant1', 'batbel1', 'baymac', 'bbwduc', 'bkcdon', 'blheag1', 'bncfly', 'bobfly1', 'brcmar1']
common_birds += ['brnowl', 'bucmot4', 'bucpar', 'bufpar', 'bunibi1', 'burowl', 'camfli1', 'chacha1', 'chbmoc1', 'chobla1', 'chvcon1', 'coffal1']
common_birds += ['compau', 'compot1', 'crbthr1', 'epaori4', 'eulfly1', 'fepowl', 'flawar1', 'fusfly1', 'gilhum1', 'giwrai1', 'grasal3', 'greani1']
common_birds += ['greant1', 'greela', 'grekis', 'gretho2', 'greyel', 'grfdov1', 'gycwor1', 'houspa', 'larela1', 'lesela1', 'limpki', 'linwoo1']
common_birds += ['litnig1', 'masgna1', 'oliwoo1', 'orwpar', 'osprey', 'pabspi1', 'paltan1', 'phecuc1', 'picpig2', 'pirfly1', 'plasla1', 'plcjay1']
common_birds += ['pvttyr1', 'rebscy1', 'recfin1', 'redjun', 'relser1', 'rinkin1', 'roahaw', 'rubthr1', 'rufgna3', 'rufhor2', 'rufnig1', 'rumfly1']
common_birds += ['rutjac1', 'saffin', 'saytan1', 'scadov1', 'schpar1', 'shcfly1', 'shtnig1', 'sibtan2', 'smbani', 'sobcac1', 'sobtyr1']
common_birds += ['socfly1', 'sofspi1', 'souant1', 'soulap1', 'spbant3', 'spispi1', 'squcuc1', 'stbwoo2', 'strcuc1', 'strher2', 'strowl1']
common_birds += ['swtman1', 'tattin1', 'thlwre1', 'toctou1', 'trokin', 'trsowl', 'undtin1', 'varant1', 'watjac1', 'wesfie1', 'wfwduc1', 'whbwar2']
common_birds += ['whiwoo1', 'whtdov', 'y00678', 'yebela1', 'yecpar', 'yeofly1']
rare_birds = ['ashgre1', 'bafcur1', 'bcwfin2', 'bkhpar', 'blchaw1', 'blttit1', 'cibspi1', 'crebec1', 'dwatin1', 'fabwre1', 'ficman1', 'fotfly']
rare_birds += ['glteme1', 'grepot1', 'grhtan1', 'horscr1', 'hyamac1', 'lesgrf1', 'litcuc2', 'mabpar', 'magant1', 'magtan2', 'nacnig1', 'ocecra1']
rare_birds += ['orbtro3', 'palhor3', 'platyr1', 'pluibi1', 'purjay1', 'ragmac1', 'rivwar1', 'rufcac2', 'rufcas2', 'ruftho1', 'ruftof1']
rare_birds += ['ruther1', 'sabspa1', 'scther1', 'shshaw', 'smbtin1', 'souscr1', 'sptnig1', 'swthum1', 'whbant2', 'whlspi1', 'whnjay1', 'whwpic1']
rare_birds += ['yebcar', 'yecmac', 'yehcar1']

# common_insects excludes 244024 because it has a separate model.

common_insects = ['47158son01', '47158son03', '47158son04', '47158son06', '47158son07', '47158son08', '47158son10', '47158son11', '47158son13']
common_insects += ['47158son17', '47158son21', '47158son22', '47158son23', '47158son24', '47158son25']
rare_insects = ['1161364', '47158son02', '47158son05', '47158son09', '47158son12', '47158son14', '47158son15', '47158son16', '47158son18']
rare_insects += ['47158son19', '47158son20', '760266']

common_mammals = ['41970', '43435', '47144', '516975', '74113']
rare_mammals = ['209233', '738183', '74580']

In [17]:
import pickle

with open(f"rare_embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

with open(f"rare_base_rates.pkl", "rb") as f:
    rare_base_rates = pickle.load(f)

In [18]:
def rare_mammal_similarity_score(rare_mammal, new_spec):
    x = new_spec[np.newaxis, ...]

    x_emb = mammal_feature_model.predict(x)
    x_emb = x_emb / (np.linalg.norm(x_emb, axis=1, keepdims=True) + 1e-8)

    sims = cosine_similarity(x_emb, embeddings[rare_mammal])[0]

    return sims.max()

def rare_bird_similarity_score(rare_bird, new_spec):
    x = new_spec[np.newaxis, ...]

    x_emb = bird_feature_model.predict(x)
    x_emb = x_emb / (np.linalg.norm(x_emb, axis=1, keepdims=True) + 1e-8)

    sims = cosine_similarity(x_emb, embeddings[rare_bird])[0]

    return sims.max()

def rare_insect_similarity_score(rare_insect, new_spec):
    x = new_spec[np.newaxis, ...]

    x_emb = insect_feature_model.predict(x)
    x_emb = x_emb / (np.linalg.norm(x_emb, axis=1, keepdims=True) + 1e-8)

    sims = cosine_similarity(x_emb, embeddings[rare_insect])[0]

    return sims.max()

def rare_amphibian_similarity_score(rare_amphibian, new_spec):
    x = new_spec[np.newaxis, ...]

    x_emb = amphibian_feature_model.predict(x)
    x_emb = x_emb / (np.linalg.norm(x_emb, axis=1, keepdims=True) + 1e-8)

    sims = cosine_similarity(x_emb, embeddings[rare_amphibian])[0]

    return sims.max()

In [19]:
# Using the similarity score and base rate, we estimate the actual probability that a given rare animal is present in the spectrogram.

def similarity_to_probability(score, base_rate, center=0.90, sharpness=30):
    prior_logit = np.log(base_rate / (1 - base_rate))

    evidence_logit = sharpness * (score - center)

    prob = 1 / (1 + np.exp(-(prior_logit + evidence_logit)))

    return prob

In [10]:
# Converts a .ogg file into a mel spectrogram that can be fed into our models.

def preprocess_audio(file_path):
    y, sr = librosa.load(file_path, sr=32000)
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    return mel_spec

In [20]:
# This block predicts how likely it is that a spectrogram belongs to each animal species.
# The classes are Insecta, Reptilia, Amphibia, Mammalia, Aves.

def predict(file_path):
    
    final_preds = {}
    
    # First, we use animal_model to determine the likelihood that the file has animals from each class.
    mel_spec = preprocess_audio(file_path)
    mel_spec = np.expand_dims(mel_spec, axis=-1)
    animal_preds = animal_model.predict(np.array([mel_spec]))[0]

    # Then, for each animal class, we use the corresponding common species model to determine the likelihood that the file has animals from the most common species in that class.
    common_bird_pred = common_bird_model.predict(np.array([mel_spec]))[0]
    common_amphibian_pred = common_amphibian_model.predict(np.array([mel_spec]))[0]
    common_insect0_pred = common_insect0_model.predict(np.array([mel_spec]))[0]
    common_insect1_pred = common_insect1_model.predict(np.array([mel_spec]))[0]
    common_mammal_pred = common_mammal_model.predict(np.array([mel_spec]))[0]

    for i in range(len(common_insects)):
        final_preds[common_insects[i]] = float(animal_preds[0] * common_insect1_pred[i])
    for i in range(len(common_amphibians)):
        final_preds[common_amphibians[i]] = float(animal_preds[2] * common_amphibian_pred[i])
    for i in range(len(common_mammals)):
        final_preds[common_mammals[i]] = float(animal_preds[3] * common_mammal_pred[i])
    for i in range(len(common_birds)):
        final_preds[common_birds[i]] = float(animal_preds[4] * common_bird_pred[i])
    
    final_preds['244024'] = float(animal_preds[0] * common_insect0_pred[0])
    final_preds['116570'] = float(animal_preds[1])
    
    for insect in rare_insects:
        final_preds[insect] = float(animal_preds[0] * similarity_to_probability(rare_insect_similarity_score(insect, mel_spec), rare_base_rates[insect], 0.90, 30))
    for amphibian in rare_amphibians:
        final_preds[amphibian] = float(animal_preds[2] * similarity_to_probability(rare_amphibian_similarity_score(amphibian, mel_spec), rare_base_rates[amphibian], 0.90, 30))
    for mammal in rare_mammals:
        final_preds[mammal] = float(animal_preds[3] * similarity_to_probability(rare_mammal_similarity_score(mammal, mel_spec), rare_base_rates[mammal], 0.90, 30))
    for bird in rare_birds:
        final_preds[bird] = float(animal_preds[4] * similarity_to_probability(rare_bird_similarity_score(bird, mel_spec), rare_base_rates[bird], 0.90, 30))
    
    return final_preds

In [22]:
import tempfile
import soundfile as sf
import pandas as pd

sample = pd.read_csv("sample_submission.csv")
species_list = list(sample.columns[1:])

rows = []
SR = 32000
CHUNK_SECONDS = 5
chunk_len = SR * CHUNK_SECONDS

for filename in os.listdir(TEST_DIR):
    if filename.endswith(".ogg"):
        file_path = os.path.join(TEST_DIR, filename)
        file_id = filename.replace(".ogg", "")

        audio, sr = librosa.load(file_path, sr=SR)

        for start_sample in range(0, len(audio), chunk_len):
            end_sample = start_sample + chunk_len
            chunk = audio[start_sample:end_sample]

            if len(chunk) < chunk_len:
                continue

            end_seconds = (start_sample // SR) + CHUNK_SECONDS
            row_id = f"{file_id}_{end_seconds}"

            with tempfile.NamedTemporaryFile(suffix=".ogg", delete=False) as tmp:
                temp_chunk_path = tmp.name

            sf.write(temp_chunk_path, chunk, SR, format="OGG")

            preds = predict(temp_chunk_path)

            row = {"row_id": row_id}

            for species in species_list:
                row[species] = float(preds.get(str(species), 0.0))

            rows.append(row)

            os.remove(temp_chunk_path)

submission = pd.DataFrame(rows, columns=sample.columns)
submission.to_csv("submission.csv", index=False)

submission.head()

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1


In [ ]:
# Submit this file to Kaggle along with the following:

#best_multilabel_model.keras
#best_common_bird_model_gpu.keras
#best_common_amphibian_model.keras
#best_insect0_model.keras
#best_insect1_model.keras
#best_common_mammal_model.keras
#rare_embeddings.pkl
#rare_base_rates.pkl

In [51]:
print(species_list)

['1161364', '116570', '1176823', '1491113', '1595929', '209233', '22930', '22956', '22961', '22967', '22973', '22983', '22985', '23150', '23154', '23158', '23176', '23724', '24279', '24285', '24287', '24321', '244024', '25073', '25092', '25214', '326272', '41970', '43435', '47144', '47158son01', '47158son02', '47158son03', '47158son04', '47158son05', '47158son06', '47158son07', '47158son08', '47158son09', '47158son10', '47158son11', '47158son12', '47158son13', '47158son14', '47158son15', '47158son16', '47158son17', '47158son18', '47158son19', '47158son20', '47158son21', '47158son22', '47158son23', '47158son24', '47158son25', '476521', '516975', '517063', '555123', '555145', '555146', '64898', '65377', '65380', '66971', '67107', '67252', '70711', '738183', '74113', '74580', '760266', 'ashgre1', 'astcra1', 'bafcur1', 'baffal1', 'banana', 'barant1', 'batbel1', 'baymac', 'bbwduc', 'bcwfin2', 'bkcdon', 'bkhpar', 'blchaw1', 'blheag1', 'blttit1', 'bncfly', 'bobfly1', 'brcmar1', 'brnowl', 'buc